In [1]:
# Cell 1: Package Installation
# Installs required Python packages using pip if they are not already installed
#!pip install dnspython ipywidgets pandas tqdm

In [2]:
# Cell 2: Library Imports Imports all necessary libraries and modules for the DNS checker
import dns.resolver
import dns.name
import dns.dnssec
import dns.message
import dns.query
import dns.rdatatype
import dns.rdataclass
import ipywidgets as widgets
from IPython.display import display, HTML, FileLink
import pandas as pd
import os
import base64
import urllib.parse
import time
import logging
import socket
from concurrent.futures import ThreadPoolExecutor, TimeoutError
from tqdm.notebook import tqdm
import html
import re
import json

In [3]:
# Cell 3: DNS Configuration Class Defines the DNSConfig class that centralizes all configuration settings
class DNSConfig:
    """Centralized configuration for DNS checking."""
    
    # DNS resolution settings
    DEFAULT_TIMEOUT = 3.0
    DEFAULT_LIFETIME = 5.0
    DEFAULT_MAX_DEPTH = 5
    
    # Concurrency settings
    DEFAULT_MAX_WORKERS = 10
    
    # Public DNS nameservers to use
    DEFAULT_NAMESERVERS = [
        '1.1.1.1',         # Cloudflare
        '8.8.8.8',         # Google
        '9.9.9.9',         # Quad9
        '94.140.14.140',   # AdGuard
        '84.200.70.40'     # DNSWATCH
    ]
    
    # Record types to check
    DEFAULT_RECORD_TYPES = ['A', 'AAAA', 'CNAME', 'CAA', 'DNSKEY', 'DS']
    
    # Directories
    @staticmethod
    def ensure_dirs():
        """Ensure required directories exist."""
        dirs = ['logs', 'exports']
        for dir_name in dirs:
            os.makedirs(os.path.join(os.getcwd(), dir_name), exist_ok=True)
        return True
    
    @staticmethod
    def setup_logging(log_file='dns_checks.log'):
        """Set up logging configuration."""
        # Ensure logs directory exists
        log_dir = os.path.join(os.getcwd(), 'logs')
        os.makedirs(log_dir, exist_ok=True)
        
        # Full path for log file
        full_log_path = os.path.join(log_dir, log_file)
        
        # Configure logging
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s: %(message)s',
            datefmt='%Y-%m-%d %H:%M:%S',
            handlers=[
                logging.FileHandler(full_log_path),
                logging.StreamHandler()  # Also log to console
            ]
        )
        return full_log_path

In [4]:
# Cell 4: DNS Cache Class Implements caching functionality to store DNS lookup results
class DNSCache:
    """Cache DNS lookup results to avoid redundant queries."""
    
    def __init__(self):
        """Initialize an empty cache."""
        self.cache = {}
        self.hit_count = 0
        self.miss_count = 0
    
    def get(self, domain, record_type):
        """Get cached result if available."""
        key = (domain.lower(), record_type)
        if key in self.cache:
            self.hit_count += 1
            return self.cache[key]
        self.miss_count += 1
        return None
    
    def store(self, domain, record_type, result):
        """Store result in cache."""
        key = (domain.lower(), record_type)
        self.cache[key] = result
    
    def get_stats(self):
        """Return cache statistics."""
        total = self.hit_count + self.miss_count
        hit_rate = (self.hit_count / total) * 100 if total > 0 else 0
        return {
            'hits': self.hit_count,
            'misses': self.miss_count,
            'total': total,
            'hit_rate': f"{hit_rate:.1f}%"
        }
    
    def clear(self):
        """Clear the cache."""
        self.cache.clear()
        self.hit_count = 0
        self.miss_count = 0

In [5]:
# Cell 5: DNS Checker Class Core functionality for checking DNS records and DNSSEC status
class DNSChecker:
    """Centralized class for DNS record checking functionality."""
    
    def __init__(self, config=None, cache=None):
        """Initialize with configuration."""
        self.config = config or DNSConfig()
        self.cache = cache or DNSCache()
    
    def get_resolver(self):
        """Create a properly configured resolver instance."""
        resolver = dns.resolver.Resolver()
        resolver.timeout = self.config.DEFAULT_TIMEOUT
        resolver.lifetime = self.config.DEFAULT_LIFETIME
        resolver.nameservers = self.config.DEFAULT_NAMESERVERS
        return resolver
    
    def resolve_record(self, domain, record_type):
        """Resolve a DNS record with caching and error handling."""
        # Check cache first
        cached_result = self.cache.get(domain, record_type)
        if cached_result:
            return cached_result
        
        # Create resolver and attempt lookup
        resolver = self.get_resolver()
        result = {
            'domain': domain,
            'record_type': record_type,
            'records': [],
            'status': 'unknown',
            'error': None
        }
        
        try:
            answers = resolver.resolve(domain, record_type)
            
            # Process results based on record type
            if record_type == 'CAA':
                result['records'] = [
                    f"{rdata.flags} {rdata.tag} \"{rdata.value}\"" 
                    for rdata in answers
                ]
            elif record_type == 'CNAME':
                result['records'] = [str(rdata.target).rstrip('.') for rdata in answers]
            else:
                result['records'] = [str(rdata) for rdata in answers]
            
            result['status'] = 'NOERROR'
            
        except dns.resolver.NoAnswer:
            result['status'] = 'NOERROR'
            result['error'] = f'No {record_type} records found'
        except dns.resolver.NXDOMAIN:
            result['status'] = 'NXDOMAIN'
            result['error'] = 'Domain does not exist'
        except dns.resolver.NoNameservers:
            result['status'] = 'SERVFAIL/REFUSED'
            result['error'] = 'DNS query refused or server failure'
        except dns.exception.Timeout:
            result['status'] = 'TIMEOUT'
            result['error'] = 'DNS query timed out'
        except Exception as e:
            result['status'] = 'ERROR'
            result['error'] = str(e)
        
        # Cache the result
        self.cache.store(domain, record_type, result)
        return result
    
    def check_dnssec(self, domain):
        """Check if DNSSEC is enabled and properly configured."""
        # Check cache
        cached_result = self.cache.get(domain, 'DNSSEC')
        if cached_result:
            return cached_result
        
        # Default result
        result = {
            'enabled': False,
            'issues': None
        }
        
        # First check for DNSKEY records
        dnskey_result = self.resolve_record(domain, 'DNSKEY')
        
        if dnskey_result['status'] == 'NOERROR' and dnskey_result['records']:
            result['enabled'] = True
        else:
            result['enabled'] = False
            result['issues'] = dnskey_result.get('error', 'No DNSKEY records found')
            self.cache.store(domain, 'DNSSEC', result)
            return result
        
        # Check for DS records at the parent zone
        domain_parts = domain.split('.')
        if len(domain_parts) > 1:
            ds_result = self.resolve_record(domain, 'DS')
            if ds_result['status'] != 'NOERROR' or not ds_result['records']:
                result['issues'] = 'DNSSEC issue: No DS records found at parent zone'
        
        # Try to validate a sample record with DNSSEC
        validator_resolver = self.get_resolver()
        validator_resolver.dnssec = True
        
        # Attempt to resolve multiple record types
        record_types = ['A', 'SOA', 'MX']
        validation_success = False
        
        for record_type in record_types:
            try:
                validator_resolver.resolve(domain, record_type)
                validation_success = True
                break
            except dns.resolver.NoAnswer:
                continue
            except dns.resolver.DNSSECValidationError as e:
                result['issues'] = f"DNSSEC validation error: {str(e)}"
                break
        
        if not validation_success and not result['issues']:
            result['issues'] = "Unable to validate DNSSEC with available records"
        
        # Cache and return result
        self.cache.store(domain, 'DNSSEC', result)
        return result
    
    def traverse_cname_chain(self, domain, max_depth=None):
        """Follow CNAME records to find the ultimate target domain."""
        if max_depth is None:
            max_depth = self.config.DEFAULT_MAX_DEPTH
        
        # Check cache
        cached_result = self.cache.get(domain, 'CNAME_CHAIN')
        if cached_result:
            return cached_result
        
        # Initialize result
        result = {
            'original_domain': domain,
            'final_domain': domain,
            'cname_chain': []
        }
        
        current_domain = domain
        
        # Follow CNAME chain
        for depth in range(max_depth):
            cname_result = self.resolve_record(current_domain, 'CNAME')
            
            if cname_result['status'] == 'NOERROR' and cname_result['records']:
                cname_target = cname_result['records'][0]
                
                # Add to chain
                result['cname_chain'].append({
                    'source': current_domain,
                    'target': cname_target
                })
                
                # Update current and final domain
                current_domain = cname_target
                result['final_domain'] = cname_target
            else:
                # No more CNAMEs or error encountered
                if cname_result['error'] and 'No CNAME records found' not in cname_result['error']:
                    result['cname_chain'].append({
                        'source': current_domain,
                        'error': cname_result['error']
                    })
                break
        
        # Cache and return result
        self.cache.store(domain, 'CNAME_CHAIN', result)
        return result
    
    def check_parent_caa_records(self, domain):
        """Check CAA records for parent domains of a given domain."""
        # Check cache
        cached_result = self.cache.get(domain, 'PARENT_CAA')
        if cached_result:
            return cached_result
        
        # Initialize result
        parent_caa_results = {}
        
        # Split domain into parts
        domain_parts = domain.split('.')
        
        # Check CAA records for progressively shorter domain names
        for i in range(1, len(domain_parts)):
            parent_domain = '.'.join(domain_parts[i:])
            
            caa_result = self.resolve_record(parent_domain, 'CAA')
            
            if caa_result['status'] == 'NOERROR' and caa_result['records']:
                parent_caa_results[parent_domain] = '\n'.join(caa_result['records'])
                # Found CAA records, stop checking parents
                break
        
        # Cache and return result
        self.cache.store(domain, 'PARENT_CAA', parent_caa_results)
        return parent_caa_results
    
    def check_parent_dnssec(self, domain):
        """Check DNSSEC status for parent domains."""
        # Check cache
        cached_result = self.cache.get(domain, 'PARENT_DNSSEC')
        if cached_result:
            return cached_result
        
        # Initialize result
        parent_dnssec_results = {}
        
        # Split domain into parts
        domain_parts = domain.split('.')
        
        # Check DNSSEC for parent domains
        for i in range(1, len(domain_parts)):
            parent_domain = '.'.join(domain_parts[i:])
            
            # Check DNSSEC
            dnssec_result = self.check_dnssec(parent_domain)
            
            parent_dnssec_results[parent_domain] = {
                'enabled': "Yes" if dnssec_result['enabled'] else "No",
                'issues': dnssec_result['issues'] if dnssec_result['issues'] else "None detected"
            }
            
            # Stop if we found a domain with DNSSEC enabled
            if dnssec_result['enabled']:
                break
        
        # Cache and return result
        self.cache.store(domain, 'PARENT_DNSSEC', parent_dnssec_results)
        return parent_dnssec_results
    
    def check_all_records(self, domain):
        """Comprehensive check of all relevant DNS records for a domain."""
        results = {
            'domain': domain
        }
        
        # Check DNSSEC status
        dnssec_result = self.check_dnssec(domain)
        results['dnssec_enabled'] = "Yes" if dnssec_result['enabled'] else "No"
        results['dnssec_issues'] = dnssec_result['issues'] if dnssec_result['issues'] else "None detected"
        
        # Traverse CNAME chain
        cname_traversal = self.traverse_cname_chain(domain)
        results['cname_traversal'] = cname_traversal
        
        # Determine domain to use for subsequent checks
        check_domain = cname_traversal.get('final_domain', domain)
        
        # Check DNSSEC for all domains in the CNAME chain
        results['cname_dnssec_checks'] = {}
        if cname_traversal.get('cname_chain'):
            for cname_hop in cname_traversal['cname_chain']:
                domains_to_check = []
                
                # Add source domain
                source = cname_hop.get('source')
                if isinstance(source, str):
                    domains_to_check.append(source)
                
                # Add target domain
                target = cname_hop.get('target')
                if isinstance(target, str):
                    domains_to_check.append(target)
                
                # Check DNSSEC for each domain
                for check_domain in domains_to_check:
                    dnssec_result = self.check_dnssec(check_domain)
                    results['cname_dnssec_checks'][check_domain] = {
                        'enabled': "Yes" if dnssec_result['enabled'] else "No",
                        'issues': dnssec_result['issues'] if dnssec_result['issues'] else "None detected"
                    }
        
        # Check DNSSEC for parent domains
        results['parent_dnssec_checks'] = self.check_parent_dnssec(domain)
        
        # Check basic DNS records
        for record_type in ['A', 'AAAA', 'CNAME', 'CAA']:
            record_result = self.resolve_record(check_domain, record_type)
            
            if record_result['records']:
                results[record_type.lower()] = '\n'.join(record_result['records'])
            else:
                results[record_type.lower()] = record_result['error'] or f'No {record_type} records found'
        
        # Check parent CAA records if no CAA found on the domain
        if 'No CAA records found' in results.get('caa', ''):
            parent_caa = self.check_parent_caa_records(check_domain)
            if parent_caa:
                results['parent_caa'] = parent_caa
        
        # Determine overall DNS status
        if any('Domain does not exist' in str(results.get(key, '')) for key in ['a', 'aaaa', 'cname', 'caa']):
            results['dns_status'] = 'NXDOMAIN - Domain does not exist'
        elif any('DNS query refused or server failure' in str(results.get(key, '')) for key in ['a', 'aaaa', 'cname', 'caa']):
            results['dns_status'] = 'SERVFAIL/REFUSED - Server failed to complete the DNS request or refused the query'
        else:
            results['dns_status'] = 'NOERROR - No DNS errors detected'
        
        return results

In [6]:
# Cell 6: Record Formatter Class Handles sanitization and formatting of DNS records for display
class RecordFormatter:
    """Handle sanitization and formatting of DNS records."""
    
    @staticmethod
    def sanitize_value(value, max_length=500):
        """Comprehensive sanitization of values to prevent XSS."""
        if not isinstance(value, str):
            return str(value)
        
        # Convert to string and normalize
        sanitized = str(value)
        
        # Remove all HTML tags
        sanitized = re.sub(r'<[^>]*>', '', sanitized)
        
        # Remove potential JS event handlers
        sanitized = re.sub(r'\bon\w+\s*=', '', sanitized, flags=re.IGNORECASE)
        
        # Remove javascript: and data: URLs
        sanitized = re.sub(r'(javascript|data|vbscript):', '', sanitized, flags=re.IGNORECASE)
        
        # HTML escape special characters
        sanitized = html.escape(sanitized)
        
        # Truncate to prevent overly long values
        sanitized = sanitized[:max_length]
        
        return sanitized
    
    @staticmethod
    def deep_decode(encoded_str, max_depth=5):
        """Recursively decode encoded strings (URL encoding, HTML entities)."""
        if not isinstance(encoded_str, str):
            return str(encoded_str)
        
        # Recursive URL decoding
        prev_str = encoded_str
        decoded_str = urllib.parse.unquote(encoded_str)
        depth = 0
        
        while decoded_str != prev_str and depth < max_depth:
            prev_str = decoded_str
            decoded_str = urllib.parse.unquote(decoded_str)
            depth += 1
        
        # Recursive HTML entity decoding
        prev_str = decoded_str
        decoded_str = html.unescape(decoded_str)
        depth = 0
        
        while decoded_str != prev_str and depth < max_depth:
            prev_str = decoded_str
            decoded_str = html.unescape(decoded_str)
            depth += 1
        
        return decoded_str
    
    @staticmethod
    def format_caa_records(caa_value):
        """Format CAA records with security classification."""
        # Default values
        if not isinstance(caa_value, str):
            return "No CAA records", False, True
        
        if 'No CAA records found' in caa_value:
            return 'No CAA records found', False, True
        
        # Initialize flags
        is_malicious = False
        contains_ssl_com = False
        
        # Check for potential malicious content
        if any(x in caa_value.lower() for x in ['script', 'alert', 'onerror']):
            return "Warning: Potential malicious code detected and redacted", True, False
        
        try:
            # Decode and process CAA value
            decoded = RecordFormatter.deep_decode(caa_value)
            
            # Process each line
            formatted_records = []
            for line in decoded.split('\n'):
                clean_line = line.strip()
                
                # Extract standard CAA format
                std_match = re.search(r'(\d+)\s+(issue|issuewild)\s+["\']([^"\']+)["\']', clean_line)
                if std_match:
                    tag = std_match.group(2)
                    domain = std_match.group(3)
                    formatted_records.append(f"{tag}: {domain}")
                    if 'ssl.com' in domain.lower():
                        contains_ssl_com = True
                    continue
                
                # Look for domains in less standard formats
                if ('issue' in clean_line or 'issuewild' in clean_line) and re.search(r'\w+\.\w+', clean_line):
                    domain_match = re.search(r'["\']?([a-zA-Z0-9*.-]+\.[a-zA-Z]{2,})["\']?', clean_line)
                    if domain_match:
                        domain = domain_match.group(1)
                        tag = 'issuewild' if 'issuewild' in clean_line else 'issue'
                        formatted_records.append(f"{tag}: {domain}")
                        if 'ssl.com' in domain.lower():
                            contains_ssl_com = True
                    continue
                
                # Handle property tags that aren't issue/issuewild
                prop_match = re.search(r'(\d+)\s+(\w+)\s+["\']([^"\']+)["\']', clean_line)
                if prop_match:
                    flag = prop_match.group(1)
                    tag = prop_match.group(2)
                    value = prop_match.group(3)
                    # Remove 'b' prefix if present
                    value = re.sub(r'^b[\'"]|[\'"]$', '', value)
                    formatted_records.append(f"{tag}: {value}")
            
            # Format output
            if formatted_records:
                return ", ".join(formatted_records), is_malicious, contains_ssl_com
            else:
                # Provide a summary if parsing failed
                cleaned = re.sub(r'\s+', ' ', decoded).strip()
                # Remove 'b' prefix if present
                cleaned = re.sub(r'b[\'"]([^\'"])[\'"]', r'\1', cleaned)
                if len(cleaned) > 80:
                    cleaned = cleaned[:80] + "..."
                
                # Check if it contains ssl.com
                if 'ssl.com' in decoded.lower():
                    contains_ssl_com = True
                
                return f"CAA value: {cleaned}", is_malicious, contains_ssl_com
        
        except Exception as e:
            return f"Error parsing CAA: {str(e)[:50]}", True, False
    
    @staticmethod
    def simplify_cname_traversal(val):
        """Simplify CNAME traversal to final domain name."""
        if isinstance(val, dict):
            if val.get('cname_chain'):
                return val.get('final_domain', 'Unknown')
            if val.get('error'):
                return 'Error'
        return 'No CNAME'
    
    @staticmethod
    def simplify_cname_dnssec_checks(val):
        """Simplify CNAME DNSSEC checks to a summary."""
        if not isinstance(val, dict):
            return 'Unknown'
        
        # Check for enabled domains
        enabled_domains = [
            domain for domain, details in val.items() 
            if details.get('enabled') == 'Yes'
        ]
        
        # Check for domains with actual configuration issues (not just disabled)
        issue_domains = [
            domain for domain, details in val.items() 
            if (details.get('enabled') == 'Yes' and 
                details.get('issues') and 
                details.get('issues') != 'None detected')
        ]
        
        if enabled_domains and not issue_domains:
            return 'DNSSEC enabled'
        elif issue_domains:
            return 'DNSSEC misconfigured'
        else:
            return 'DNSSEC not enabled'
    
    @staticmethod
    def format_dnssec_status(enabled, issues):
        """Format DNSSEC status in a standardized way."""
        if not isinstance(enabled, str):
            enabled = str(enabled)
        
        status = "Enabled" if enabled == "Yes" else "Not Enabled"
        
        if not issues or issues == "None detected":
            issue_text = "None"
        else:
            # Simplify common issues
            if "No DNSKEY records found" in issues:
                issue_text = "No DNSKEY records"
            elif "Domain does not exist" in issues:
                issue_text = "Domain does not exist"
            elif "DNS query timed out" in issues:
                issue_text = "See status"
            elif "DNS query refused" in issues:
                issue_text = "See status"
            else:
                issue_text = issues[:50] + "..." if len(issues) > 50 else issues
        
        return f"Status: {status}, Issues: {issue_text}"
    
    @staticmethod
    def format_dns_status(status):
        """Format DNS status in a standardized way."""
        if not isinstance(status, str):
            return "Unknown"
        
        # Extract main status code
        if "NOERROR" in status:
            code = "NOERROR"
            details = "No errors"
        elif "NXDOMAIN" in status:
            code = "NXDOMAIN"
            details = "Domain does not exist"
        elif "SERVFAIL" in status or "REFUSED" in status:
            code = "SERVFAIL/REFUSED"
            details = "Server error or query refused"
        elif "TIMEOUT" in status:
            code = "TIMEOUT"
            details = "Query timed out"
        else:
            code = "UNKNOWN"
            details = status
        
        return f"Status: {code}, Details: {details}"
    
    @staticmethod
    def format_ip_records(a_records, aaaa_records):
        """Combine A and AAAA records into a single field."""
        a_valid = isinstance(a_records, str) and "No A records found" not in a_records and "Domain does not exist" not in a_records
        aaaa_valid = isinstance(aaaa_records, str) and "No AAAA records found" not in aaaa_records and "Domain does not exist" not in aaaa_records
        
        if a_valid and aaaa_valid:
            return f"IPv4: {a_records}, IPv6: {aaaa_records}"
        elif a_valid:
            return f"IPv4: {a_records}"
        elif aaaa_valid:
            return f"IPv6: {aaaa_records}"
        elif "Domain does not exist" in str(a_records) or "Domain does not exist" in str(aaaa_records):
            return "See status"
        elif "DNS query" in str(a_records) or "DNS query" in str(aaaa_records):
            return "See status"
        else:
            return "No IP records"
    
    @staticmethod
    def simplify_parent_dnssec_checks(val):
        """Simplify parent DNSSEC checks to a summary of the most relevant domain."""
        if not isinstance(val, dict):
            return 'na'
        
        # Sort domains from most specific to most general
        sorted_domains = sorted(
            val.keys(), 
            key=lambda x: len(x.split('.')), 
            reverse=True
        )
        
        # Check each domain starting with most specific
        for domain in sorted_domains:
            # Skip TLDs
            domain_parts = domain.split('.')
            if len(domain_parts) <= 2:
                continue
            
            # Get base domain (e.g., example.com)
            base_domain = '.'.join(domain_parts[-2:])
            
            # Check DNSSEC status
            details = val.get(domain, {})
            if details.get('enabled') == 'Yes':
                return f"{base_domain}:enabled"
            elif details.get('enabled') == 'No':
                return f"{base_domain}:disabled"
        
        return 'na'
    
    @staticmethod
    def simplify_parent_caa(val):
        """Simplify parent CAA records to a summary."""
        if not isinstance(val, dict):
            return 'No parent CAA'
        
        # Process first parent with CAA records
        for domain, records in val.items():
            if isinstance(records, str):
                # Decode and extract CAA info
                decoded = RecordFormatter.deep_decode(records)
                
                # Remove 'b' prefix if present
                decoded = re.sub(r'b[\'"]([^\'"])[\'"]', r'\1', decoded)
                
                # Extract CAA using regex
                caa_info = []
                for match in re.finditer(r'(\d+)\s+(issue|issuewild)\s+["\']([^"\']+)["\']', decoded):
                    tag = match.group(2)
                    value = match.group(3)
                    caa_info.append(f"{tag}: {value}")
                
                if caa_info:
                    return f"{domain}: {', '.join(caa_info)}"
                
                # Fallback to simplified text
                cleaned = re.sub(r'\s+', ' ', decoded).strip()
                if len(cleaned) > 80:
                    cleaned = cleaned[:80] + "..."
                
                return f"{domain}: {cleaned}"
            
            elif isinstance(records, list):
                decoded_records = [RecordFormatter.deep_decode(r) for r in records]
                return f"{domain}: {' | '.join(decoded_records)}"
        
        return 'No parent CAA'

In [7]:
# Cell 7: DataFrame Styler Class Manages styling and visual formatting of DNS check results
class DataFrameStyler:
    """Handle styling and formatting of DNS check results DataFrame."""
    
    def __init__(self, df, formatter=None):
        """Initialize with a DataFrame and optional formatter."""
        self.df = df.copy()
        self.formatter = formatter or RecordFormatter()
        self.caa_flags = {}
        self.dns_status = {}
    
    def prepare_for_display(self):
        """Prepare DataFrame for display by simplifying complex columns and grouping related data."""
        # Create column groups
        self.column_groups = {
            'Domain Information': ['domain', 'dns_status'],
            'DNSSEC Information': ['dnssec_status', 'parent_dnssec_checks'],
            'CAA Information': ['caa', 'parent_caa'],
            'CNAME Information': ['cname_traversal', 'cname_dnssec_checks'],
            'DNS Records': ['ip_records']
        }
        
        # Combine A and AAAA records
        if 'a' in self.df.columns and 'aaaa' in self.df.columns:
            self.df['ip_records'] = self.df.apply(
                lambda row: self.formatter.format_ip_records(row['a'], row['aaaa']),
                axis=1
            )
        
        # Combine DNSSEC enabled and issues
        if 'dnssec_enabled' in self.df.columns and 'dnssec_issues' in self.df.columns:
            self.df['dnssec_status'] = self.df.apply(
                lambda row: self.formatter.format_dnssec_status(row['dnssec_enabled'], row['dnssec_issues']),
                axis=1
            )
            # Store original values for styling
            self.dns_status['dnssec_enabled'] = self.df['dnssec_enabled']
            self.dns_status['dnssec_issues'] = self.df['dnssec_issues']
        
        # Format DNS status
        if 'dns_status' in self.df.columns:
            # Store original values for styling
            self.dns_status['original'] = self.df['dns_status']
            # Apply standardized formatting
            self.df['dns_status'] = self.df['dns_status'].apply(self.formatter.format_dns_status)
        
        # Process CAA column with special handling
        if 'caa' in self.df.columns:
            # Get formatted CAA and store flags for styling
            results = self.df['caa'].apply(self.formatter.format_caa_records)
            
            # Store formatted value and flags
            self.df['caa'] = results.apply(lambda x: x[0])
            self.caa_flags = {
                'malicious': results.apply(lambda x: x[1]),
                'ssl_com': results.apply(lambda x: x[2])
            }
        
        # Simplify complex nested columns
        simplify_mappings = {
            'cname_traversal': self.formatter.simplify_cname_traversal,
            'cname_dnssec_checks': self.formatter.simplify_cname_dnssec_checks,
            'parent_dnssec_checks': self.formatter.simplify_parent_dnssec_checks
        }
        
        for col, func in simplify_mappings.items():
            if col in self.df.columns:
                self.df[col] = self.df[col].apply(func)
        
        # Process parent CAA if present
        if 'parent_caa' in self.df.columns:
            self.df['parent_caa'] = self.df['parent_caa'].apply(self.formatter.simplify_parent_caa)
        
        # Drop redundant columns
        columns_to_drop = []
        if 'dnssec_enabled' in self.df.columns and 'dnssec_status' in self.df.columns:
            columns_to_drop.extend(['dnssec_enabled', 'dnssec_issues'])
        
        if 'ip_records' in self.df.columns:
            columns_to_drop.extend(['a', 'aaaa'])
        
        if 'cname' in self.df.columns and 'cname_traversal' in self.df.columns:
            columns_to_drop.append('cname')
        
        # Actually drop the columns
        self.df = self.df.drop(columns=columns_to_drop, errors='ignore')
        
        return self.df
    
    def style_dns_status(self, val):
        """Style DNS status with color coding."""
        if not isinstance(val, str):
            return ''
        
        if 'NOERROR' in val:
            return 'background-color: #2ECC71; color: white; font-weight: bold;'
        elif 'NXDOMAIN' in val or 'SERVFAIL' in val or 'REFUSED' in val or 'ERROR' in val:
            return 'background-color: #FF6B6B; color: white; font-weight: bold;'
        elif 'TIMEOUT' in val:
            return 'background-color: #F39C12; color: white; font-weight: bold;'
        return ''
    
    def style_dnssec_status(self, val):
        """Style DNSSEC status with color coding."""
        if not isinstance(val, str):
            return ''
        
        if 'Status: Enabled' in val:
            return 'background-color: #3498DB; color: white; font-weight: bold;'
        elif 'Issues: None' not in val and 'Status: Not Enabled' in val:
            return 'background-color: #F39C12; color: white; font-weight: bold;'
        elif 'misconfigured' in val.lower() or ('Issues:' in val and 'None' not in val):
            return 'background-color: #FF6B6B; color: white; font-weight: bold;'
        return ''
    
    def style_caa_column(self, v, i):
        """Style CAA column based on previously stored flags."""
        if not isinstance(v, str):
            return ''
            
        if 'See status' in v:
            return 'background-color: #ECECEC; color: #666666;'
            
        if 'malicious' in self.caa_flags and i < len(self.caa_flags['malicious']):
            if self.caa_flags['malicious'].iloc[i]:
                return 'background-color: #FF6B6B; color: white; font-weight: bold;'  # Red for malicious
            elif self.caa_flags['ssl_com'].iloc[i]:
                return 'background-color: #2ECC71; color: white; font-weight: bold;'  # Green for ssl.com
            elif 'No CAA records found' in str(v):
                return 'background-color: #2ECC71; color: white; font-weight: bold;'  # Green for no records
            else:
                return 'background-color: #FF6B6B; color: white; font-weight: bold;'  # Red for other CAs
        return ''
    
    def style_see_status(self, val):
        """Style cells that reference the status column."""
        if isinstance(val, str) and 'See status' in val:
            return 'background-color: #ECECEC; color: #666666;'
        return ''
    
    def get_group_styles(self):
        """Generate CSS styles for column groups."""
        group_styles = []
        
        # Define colors for each group
        group_colors = {
            'Domain Information': '#4CAF50',
            'DNSSEC Information': '#3498DB',
            'CAA Information': '#9B59B6',
            'CNAME Information': '#F39C12',
            'DNS Records': '#1ABC9C'
        }
        
        # Create selector for each group
        for group, color in group_colors.items():
            # Get column indices for this group
            if hasattr(self, 'column_groups'):
                cols = self.column_groups.get(group, [])
                for col in cols:
                    if col in self.df.columns:
                        col_idx = self.df.columns.get_loc(col)
                        selector = f'th.col{col_idx}'
                        style = {
                            'selector': selector,
                            'props': [
                                ('background-color', color),
                                ('color', 'white'),
                                ('font-weight', 'bold'),
                                ('border-bottom', f'3px solid {color}')
                            ]
                        }
                        group_styles.append(style)
        
        return group_styles
    
    def apply_styling(self):
        """Apply all styling rules to the DataFrame."""
        # Apply base styling
        styled_df = self.df.style
        
        # Apply column-specific stylings - FIXED HERE
        if 'dns_status' in self.df.columns:
            styled_df = styled_df.apply(
                lambda s: [self.style_dns_status(x) for x in s] if s.name == 'dns_status' else [''] * len(s),
                axis=0
            )
        
        if 'dnssec_status' in self.df.columns:
            styled_df = styled_df.apply(
                lambda s: [self.style_dnssec_status(x) for x in s] if s.name == 'dnssec_status' else [''] * len(s),
                axis=0
            )
        
        # Apply CAA styling if we have flags
        if 'caa' in self.df.columns and self.caa_flags:
            # Use a function to apply CAA styling
            styled_df = styled_df.apply(
                lambda x: [self.style_caa_column(v, i) for i, v in enumerate(x)] if x.name == 'caa' else [''] * len(x),
                axis=0
            )
        
        # Style "See status" references
        styled_df = styled_df.map(self.style_see_status)
        
        # Apply general table properties
        styled_df = styled_df.set_properties(**{
            'white-space': 'pre-wrap',
            'text-align': 'left',
            'max-width': '250px',
            'overflow': 'hidden',
            'text-overflow': 'ellipsis'
        })
        
        # Apply table header styles
        group_styles = self.get_group_styles()
        base_header_style = [{
            'selector': 'th',
            'props': [('background-color', '#4CAF50'), ('color', 'white'), ('font-weight', 'bold')]
        }]
        
        styled_df = styled_df.set_table_styles(base_header_style + group_styles)
        
        # Custom hover styling for expandable content
        styled_df = styled_df.set_table_styles([{
            'selector': 'td',
            'props': [
                ('cursor', 'pointer'),
                ('transition', 'max-width 0.3s')
            ]
        }], overwrite=False)
        
        # Generate summary statistics HTML
        if hasattr(self, 'dns_status') and 'dnssec_enabled' in self.dns_status:
            # Calculate stats
            total_domains = len(self.df)
            dnssec_enabled = sum(self.dns_status['dnssec_enabled'] == 'Yes')
            caa_records = sum(~self.df['caa'].str.contains('No CAA records found', na=False)) if 'caa' in self.df.columns else 0
            error_domains = 0
            if 'original' in self.dns_status:
                error_domains = sum(~self.dns_status['original'].str.contains('NOERROR', na=False))
            cname_usage = sum(self.df['cname_traversal'] != 'No CNAME') if 'cname_traversal' in self.df.columns else 0
            
            # Generate HTML for stats
            stats_html = f"""
           <div style="margin: 10px 0; padding: 15px; background-color: #212529; border-radius: 5px; border: 1px solid #343a40; color: #e9ecef;">
                        <h3 style="margin-top: 0; color: #8bc34a;">DNS Summary Statistics</h3>
                        <div style="display: flex; flex-wrap: wrap; gap: 20px;">
                    <div>
                        <h4>DNSSEC Adoption</h4>
                        <p>{dnssec_enabled}/{total_domains} domains have DNSSEC enabled ({dnssec_enabled/total_domains*100:.1f}%)</p>
                    </div>
                    <div>
                        <h4>CAA Implementation</h4>
                        <p>{caa_records}/{total_domains} domains have CAA records ({caa_records/total_domains*100:.1f}%)</p>
                    </div>
                    <div>
                        <h4>Error States</h4>
                        <p>{error_domains}/{total_domains} domains have DNS errors ({error_domains/total_domains*100:.1f}%)</p>
                    </div>
                    <div>
                        <h4>CNAME Usage</h4>
                        <p>{cname_usage}/{total_domains} domains use CNAME records ({cname_usage/total_domains*100:.1f}%)</p>
                    </div>
                </div>
            </div>
            """
            # Add caption with stats
            styled_df = styled_df.set_caption(stats_html)
        
        # Ensure HTML is escaped for display
        styled_df = styled_df.format(escape="html")
        
        return styled_df
    
    def get_styled_df(self):
        """Prepare and style DataFrame for display."""
        self.prepare_for_display()
        return self.apply_styling()

In [8]:
# Cell 8: UI Utilities Class Provides utility functions for UI interactions and file handling
class UIUtils:
    """Utility functions for UI interactions and file handling."""
    
    @staticmethod
    def create_download_link(df, title="Download CSV", filename="dns_check_results.csv"):
        """Create downloadable link for DataFrame."""
        try:
            # Convert DataFrame to CSV
            csv = df.to_csv(index=False)
            
            # Encode to base64
            b64 = base64.b64encode(csv.encode())
            payload = b64.decode()
            
            # Create HTML link
            html = f'<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
            
            return HTML(html)
        except Exception as e:
            logging.error(f"Error creating download link: {e}")
            return HTML(f"Error creating download link: {e}")
    
    # Add this method to the UIUtils class
    @staticmethod
    def export_html(styled_df, title="Download HTML", filename="dns_check_results.html"):
        """Create downloadable link for styled HTML table."""
        try:
            # Convert styled DataFrame to HTML
            html_content = styled_df.to_html()
            
            # Add CSS for expandable content
            expandable_css = """
            <style>
                td {
                    max-width: 250px;
                    overflow: hidden;
                    text-overflow: ellipsis;
                    white-space: nowrap;
                    transition: all 0.3s;
                }
                td:hover {
                    max-width: none !important;
                    white-space: normal !important;
                    overflow: visible !important;
                }
                .collapsible {
                    cursor: pointer;
                    background-color: #f1f1f1;
                    margin: 5px 0;
                    padding: 8px 15px;
                    width: 100%;
                    border: none;
                    text-align: left;
                    outline: none;
                    font-weight: bold;
                }
                .active, .collapsible:hover {
                    background-color: #e0e0e0;
                }
                .content {
                    padding: 0 18px;
                    max-height: 0;
                    overflow: hidden;
                    transition: max-height 0.3s ease-out;
                    background-color: #ffffff;
                }
            </style>
            <script>
                document.addEventListener('DOMContentLoaded', function() {
                    var coll = document.getElementsByClassName("collapsible");
                    for (var i = 0; i < coll.length; i++) {
                        coll[i].addEventListener("click", function() {
                            this.classList.toggle("active");
                            var content = this.nextElementSibling;
                            if (content.style.maxHeight) {
                                content.style.maxHeight = null;
                            } else {
                                content.style.maxHeight = content.scrollHeight + "px";
                            }
                        });
                    }
                });
            </script>
            """
            
            # Combine CSS and HTML
            full_html = f"<html><head>{expandable_css}</head><body>{html_content}</body></html>"
            
            # Encode to base64
            b64 = base64.b64encode(full_html.encode())
            payload = b64.decode()
            
            # Create HTML link
            download_link = f'<a download="{filename}" href="data:text/html;base64,{payload}" target="_blank">{title}</a>'
            
            return HTML(download_link)
        except Exception as e:
            logging.error(f"Error creating HTML download link: {e}")
            return HTML(f"Error creating HTML download link: {e}")

In [9]:
# Cell 9: UI Components Class Enhanced UI components for the DNS checker interface
class UIComponents:
    """Enhanced UI components for DNS checker."""
    
    @staticmethod
    def create_download_options(df, styled_df):
        """Create download options with multiple formats."""
        # Import widget dependencies
        import ipywidgets as widgets
        
        # Create download section header
        download_header = widgets.HTML(
            """<div style="background-color: #4CAF50; color: white; padding: 10px; 
                          margin: 10px 0; border-radius: 5px; font-weight: bold;">
                 Download Options
               </div>"""
        )
        
        # Create CSV download link as HTML widget
        csv_link = UIUtils.create_download_link(df, "Download CSV", "dns_results.csv")
        csv_widget = widgets.HTML(value=csv_link.data)
        
        # Create HTML download link as HTML widget
        html_link = UIUtils.export_html(styled_df, "Download HTML", "dns_results.html")
        html_widget = widgets.HTML(value=html_link.data)
        
        # Create download buttons
        download_container = widgets.HBox([
            widgets.VBox([
                widgets.HTML(value='<b>Raw Data:</b>'),
                csv_widget
            ]),
            widgets.VBox([
                widgets.HTML(value='<b>Formatted Table:</b>'),
                html_widget
            ])
        ])
    
        return widgets.VBox([download_header, download_container])
    
    @staticmethod
    def create_filter_widgets(df):
        """Create filter widgets for the DataFrame."""
        # Import widgets for filtering
        import ipywidgets as widgets
        from IPython.display import display, HTML
        
        # Container for all filters
        filters_container = widgets.VBox()
        
        # Create filter section header
        filter_header = widgets.HTML(
            """<div style="background-color: #4CAF50; color: white; padding: 10px; 
                          margin-bottom: 10px; border-radius: 5px; font-weight: bold;">
                 Filter Results
               </div>"""
        )
        
        # Create domain filter
        domain_filter = widgets.Text(
            description='Domain:',
            placeholder='Filter by domain name...',
            style={'description_width': '100px'}
        )
        
        # Create DNSSEC filter
        dnssec_options = ['All', 'Enabled', 'Not Enabled', 'Misconfigured']
        dnssec_filter = widgets.Dropdown(
            options=dnssec_options,
            value='All',
            description='DNSSEC:',
            style={'description_width': '100px'}
        )
        
        # Create DNS Status filter
        status_options = ['All', 'NOERROR', 'NXDOMAIN', 'SERVFAIL/REFUSED', 'TIMEOUT']
        status_filter = widgets.Dropdown(
            options=status_options,
            value='All',
            description='DNS Status:',
            style={'description_width': '100px'}
        )
        
        # Create CAA filter
        caa_options = ['All', 'Has CAA', 'No CAA', 'SSL.com', 'Other CA']
        caa_filter = widgets.Dropdown(
            options=caa_options,
            value='All',
            description='CAA Records:',
            style={'description_width': '100px'}
        )
        
        # Apply button
        apply_button = widgets.Button(
            description='Apply Filters',
            button_style='success',
            icon='filter'
        )
        
        # Reset button
        reset_button = widgets.Button(
            description='Reset Filters',
            button_style='warning',
            icon='refresh'
        )
        
        # Results counter
        results_count = widgets.HTML(value=f"<b>Showing all {len(df)} domains</b>")
        
        # Create button container
        button_container = widgets.HBox([apply_button, reset_button, results_count])
        
        # Add all widgets to container
        filters_container.children = [
            filter_header,
            domain_filter,
            widgets.HBox([dnssec_filter, status_filter, caa_filter]),
            button_container
        ]
        
        # Define filter application function
        def apply_filters(b):
            # Get filter values
            domain_val = domain_filter.value.lower()
            dnssec_val = dnssec_filter.value
            status_val = status_filter.value
            caa_val = caa_filter.value
            
            # Create a copy of the original DataFrame
            filtered_df = df.copy()
            
            # Apply filters
            if domain_val:
                filtered_df = filtered_df[filtered_df['domain'].str.lower().str.contains(domain_val)]
            
            if dnssec_val != 'All':
                if 'dnssec_status' in filtered_df.columns:
                    if dnssec_val == 'Enabled':
                        filtered_df = filtered_df[filtered_df['dnssec_status'].str.contains('Status: Enabled')]
                    elif dnssec_val == 'Not Enabled':
                        filtered_df = filtered_df[filtered_df['dnssec_status'].str.contains('Status: Not Enabled')]
                    elif dnssec_val == 'Misconfigured':
                        filtered_df = filtered_df[
                            (filtered_df['dnssec_status'].str.contains('Status: Enabled')) & 
                            (~filtered_df['dnssec_status'].str.contains('Issues: None'))
                        ]
            
            if status_val != 'All':
                if 'dns_status' in filtered_df.columns:
                    filtered_df = filtered_df[filtered_df['dns_status'].str.contains(status_val)]
            
            if caa_val != 'All':
                if 'caa' in filtered_df.columns:
                    if caa_val == 'Has CAA':
                        filtered_df = filtered_df[~filtered_df['caa'].str.contains('No CAA records found')]
                    elif caa_val == 'No CAA':
                        filtered_df = filtered_df[filtered_df['caa'].str.contains('No CAA records found')]
                    elif caa_val == 'SSL.com':
                        filtered_df = filtered_df[filtered_df['caa'].str.lower().str.contains('ssl.com')]
                    elif caa_val == 'Other CA':
                        filtered_df = filtered_df[
                            (~filtered_df['caa'].str.contains('No CAA records found')) & 
                            (~filtered_df['caa'].str.lower().str.contains('ssl.com'))
                        ]
            
            # Update results counter
            results_count.value = f"<b>Showing {len(filtered_df)} of {len(df)} domains</b>"
            
            # Create styled version of filtered DF
            styler = DataFrameStyler(filtered_df)
            styled_df = styler.get_styled_df()
            
            # Clear previous output
            output.clear_output()
            
            # Display results
            with output:
                display(filters_container)
                display(styled_df)
                
                # Show download options
                print("\nDownload options:")
                display(UIUtils.create_download_link(filtered_df, "Download CSV", "filtered_dns_results.csv"))
                display(UIUtils.export_html(styled_df, "Download HTML", "filtered_dns_results.html"))
        
        # Define reset function
        def reset_filters(b):
            domain_filter.value = ''
            dnssec_filter.value = 'All'
            status_filter.value = 'All'
            caa_filter.value = 'All'
            
            # Reset results counter
            results_count.value = f"<b>Showing all {len(df)} domains</b>"
            
            # Create styled version of original DF
            styler = DataFrameStyler(df)
            styled_df = styler.get_styled_df()
            
            # Clear previous output
            output.clear_output()
            
            # Display results
            with output:
                display(filters_container)
                display(styled_df)
                
                # Show download options
                print("\nDownload options:")
                display(UIUtils.create_download_link(df, "Download CSV", "dns_results.csv"))
                display(UIUtils.export_html(styled_df, "Download HTML", "dns_results.html"))
        
        # Connect handlers
        apply_button.on_click(apply_filters)
        reset_button.on_click(reset_filters)
        
        # Create output area
        output = widgets.Output()
        
        # Return widgets
        return {
            'filters': filters_container,
            'output': output,
            'apply': apply_filters,
            'reset': reset_filters
        }
    
    @staticmethod
    def create_collapsible_sections(styled_df):
        """Create collapsible sections for column groups."""
        # Import required libraries
        from IPython.display import display, HTML
        import ipywidgets as widgets
        
        # Define column groups
        column_groups = {
            'Domain Information': ['domain', 'dns_status'],
            'DNSSEC Information': ['dnssec_status', 'parent_dnssec_checks'],
            'CAA Information': ['caa', 'parent_caa'],
            'CNAME Information': ['cname_traversal', 'cname_dnssec_checks'],
            'DNS Records': ['ip_records']
        }
        
        # Create collapsible sections
        collapsible_html = """
        <style>
            .collapsible-section {
                margin-bottom: 10px;
                border-radius: 5px;
                overflow: hidden;
            }
            .collapsible-header {
                cursor: pointer;
                padding: 10px 15px;
                font-weight: bold;
                color: white;
            }
            .collapsible-content {
                display: block;
                padding: 10px;
                border: 1px solid #ddd;
                border-top: none;
            }
            .domain-section .collapsible-header { background-color: #4CAF50; }
            .dnssec-section .collapsible-header { background-color: #3498DB; }
            .caa-section .collapsible-header { background-color: #9B59B6; }
            .cname-section .collapsible-header { background-color: #F39C12; }
            .records-section .collapsible-header { background-color: #1ABC9C; }
        </style>
        <script>
            function toggleSection(sectionId) {
                var content = document.getElementById('content-' + sectionId);
                if (content.style.display === 'none') {
                    content.style.display = 'block';
                } else {
                    content.style.display = 'none';
                }
            }
        </script>
        """
        
        # Initialize HTML content with script
        html_content = collapsible_html
        
        # Loop through groups
        for i, (group_name, columns) in enumerate(column_groups.items()):
            group_class = group_name.lower().replace(' ', '-') + '-section'
            
            # Create collapsible section header
            html_content += f"""
            <div class="collapsible-section {group_class}">
                <div class="collapsible-header" onclick="toggleSection({i})">
                    {group_name}
                </div>
                <div class="collapsible-content" id="content-{i}">
            """
            
            # Add columns for this group
            for col in columns:
                if col in styled_df.columns:
                    # Get subset of styled DataFrame for this column
                    col_df = styled_df[[col]]
                    html_content += col_df.to_html()
            
            # Close section
            html_content += """
                </div>
            </div>
            """
        
        return HTML(html_content)
    
    @staticmethod
    def create_responsive_view(df):
        """Create a responsive view that shows different columns based on screen size."""
        # Import required libraries
        from IPython.display import display, HTML
        
        # Define column priorities
        priority_columns = ['domain', 'dnssec_status', 'caa', 'dns_status']
        secondary_columns = ['cname_traversal', 'parent_caa']
        tertiary_columns = ['parent_dnssec_checks', 'ip_records']
        
        # Create HTML with responsive breakpoints
        responsive_html = """
        <style>
            /* Base styles */
            .responsive-table {
                width: 100%;
                border-collapse: collapse;
            }
            .responsive-table th, .responsive-table td {
                padding: 8px;
                text-align: left;
                border: 1px solid #ddd;
            }
            .responsive-table th {
                background-color: #4CAF50;
                color: white;
            }
            
            /* Responsive breakpoints */
            @media screen and (max-width: 1200px) {
                .tertiary-column {
                    display: none;
                }
            }
            @media screen and (max-width: 800px) {
                .secondary-column {
                    display: none;
                }
            }
            @media screen and (max-width: 500px) {
                .responsive-table th, .responsive-table td {
                    padding: 4px;
                    font-size: 12px;
                }
            }
        </style>
        <table class="responsive-table">
            <thead>
                <tr>
        """
        
        # Add column headers with appropriate classes
        all_columns = priority_columns + secondary_columns + tertiary_columns
        for col in all_columns:
            if col in df.columns:
                col_class = ""
                if col in tertiary_columns:
                    col_class = "tertiary-column"
                elif col in secondary_columns:
                    col_class = "secondary-column"
                
                responsive_html += f'<th class="{col_class}">{col}</th>'
        
        responsive_html += """
                </tr>
            </thead>
            <tbody>
        """
        
        # Add rows
        for idx, row in df.iterrows():
            responsive_html += "<tr>"
            
            for col in all_columns:
                if col in df.columns:
                    col_class = ""
                    if col in tertiary_columns:
                        col_class = "tertiary-column"
                    elif col in secondary_columns:
                        col_class = "secondary-column"
                    
                    responsive_html += f'<td class="{col_class}">{row[col]}</td>'
            
            responsive_html += "</tr>"
        
        responsive_html += """
            </tbody>
        </table>
        <div style="margin-top: 10px; font-style: italic;">
            Note: Table columns automatically adjust based on screen size.
        </div>
        """
        
        return HTML(responsive_html)

In [10]:
# Cell 10: DNS Checks Runner Function  Core function that runs DNS checks for a list of domains
def run_dns_checks(domains, output_widget=None):
    """Run DNS checks for a list of domains with UI updates."""
    # Set up logging
    log_file = DNSConfig.setup_logging()
    
    # Initialize checker
    checker = DNSChecker()
    
    # Log start of checks
    logging.info(f"Starting DNS checks for {len(domains)} domain(s)")
    print(f"Checking DNS records for {len(domains)} domain(s)...")
    
    # Process domains
    results = []
    
    # Create progress bar
    with tqdm(total=len(domains), desc="DNS Checks", unit="domain") as pbar:
        for domain in domains:
            try:
                # Check domain
                result = checker.check_all_records(domain)
                results.append(result)
            except Exception as e:
                logging.error(f"Error checking {domain}: {e}")
                results.append({
                    'domain': domain, 
                    'dns_status': f'ERROR - {str(e)}',
                    'dnssec_enabled': 'Unknown',
                    'dnssec_issues': str(e)
                })
            
            # Update progress
            pbar.update(1)
            
            # Small delay to prevent overwhelming DNS servers
            time.sleep(0.1)
    
    # Convert to DataFrame
    df = pd.DataFrame(results)
    
    return {
        'df': df,
        'log_file': log_file
    }

# Attach the function to UIUtils class
UIUtils.run_dns_checks = run_dns_checks

In [11]:
# Cell 11: Main UI Setup and Handler Functions Sets up the enhanced UI with event handlingoutput = widgets.Output()

def run_enhanced_dns_checks_handler(b):
    """Enhanced handler to run DNS checks with improved UI and formatting."""
    # Clear previous output
    output.clear_output()
    
    with output:
        try:
            # Validate domain input
            if not domains_text.value.strip():
                print("Please enter at least one domain to check.")
                return
            
            # Parse domains, ignoring comments and empty lines
            domains = [
                domain.strip() 
                for domain in domains_text.value.split('\n') 
                if domain.strip() and not domain.strip().startswith('#')
            ]
            
            # Validate domains
            if not domains:
                print("No valid domains found. Please enter at least one valid domain.")
                return
            
            # Run checks
            results = UIUtils.run_dns_checks(domains, output)
            
            if results:
                # Create styled version
                styler = DataFrameStyler(results['df'])
                styled_df = styler.get_styled_df()
                
                # Create filter widgets
                filter_widgets = UIComponents.create_filter_widgets(results['df'])
                
                # Create download options
                download_options = UIComponents.create_download_options(results['df'], styled_df)
                
                # Display widgets in this order:
                # 1. Filter widgets
                display(filter_widgets['filters'])
                # 2. Download options (moved above the table)
                display(download_options)
                # 3. Table
                display(styled_df)
                
                # Store widgets in output for filter operations
                filter_widgets['output'] = output
                
                # Show completion message
                print(f"\nDNS check completed! Detailed log saved to: {results['log_file']}")
        
        except Exception as e:
            import traceback
            print(f"Unexpected error: {e}")
            traceback.print_exc()


# Add a run function that replaces the existing run_dns_checks_handler
def setup_enhanced_ui():
    """Set up the enhanced UI for DNS checking."""
    global domains_text, output  # Make both domains_text and output global
    
    # Create output widget first
    output = widgets.Output()
    
    # Ensure necessary directories exist
    DNSConfig.ensure_dirs()
    
    # Domain Input Widget
    domains_text = widgets.Textarea(
         placeholder='Enter domains (one per line)\ne.g.:\nexample.com\ngoogle.com\nd2cmedia.ca',
    layout=widgets.Layout(width='50%', height='200px'),
    style={'background-color': '#333', 'color': 'white'}  # Add these styles for dark mode
    )
    
    # Button to run checks
    run_button = widgets.Button(
        description='Check DNS Records',
        button_style='success',
        tooltip='Click to check DNS records for the domains',
        icon='check'
    )
    
    # Add header and description
    header = widgets.HTML(
        """<div style="background-color: #4CAF50; color: white; padding: 15px; 
                      margin-bottom: 20px; border-radius: 5px;">
             <h2 style="margin: 0;">Enhanced DNS & CAA Record Checker</h2>
             <p style="margin: 5px 0 0 0;">Check DNS configurations, DNSSEC status, and CAA records for multiple domains</p>
           </div>"""
    )
    
    # Instructions
    instructions = widgets.HTML(
        """<div style="margin-bottom: 15px; padding: 10px; background-color: #333333; color: #ffffff; border-radius: 5px; border: 1px solid #555555;">
             <h3 style="margin-top: 0; color: #ffffff;">Instructions:</h3>
             <ol style="color: #ffffff;">
               <li>Enter one domain per line in the text area below</li>
               <li>Click "Check DNS Records" to analyze configurations</li>
               <li>Use the filters to narrow down results</li>
               <li>Hover over cells to see full content</li>
               <li>Download results in CSV or HTML format</li>
             </ol>
           </div>"""
    )
    
    # Connect handler
    run_button.on_click(run_enhanced_dns_checks_handler)
    
    # Display UI components
    display(header)
    display(instructions)
    display(domains_text)
    display(run_button)
    display(output)
    
    # Return components for access
    return {
        'domains_text': domains_text,
        'run_button': run_button,
        'output': output
    }
# Call this function instead of the individual UI components in the main cell
ui_components = setup_enhanced_ui()

HTML(value='<div style="background-color: #4CAF50; color: white; padding: 15px; \n                      margin…

HTML(value='<div style="margin-bottom: 15px; padding: 10px; background-color: #333333; color: #ffffff; border-…

Textarea(value='', layout=Layout(height='200px', width='50%'), placeholder='Enter domains (one per line)\ne.g.…

Button(button_style='success', description='Check DNS Records', icon='check', style=ButtonStyle(), tooltip='Cl…

Output()